In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/dataset.csv")

if 'store_id' in df.columns:
    df = df.drop(columns=['store_id'])

if 'subtotal' in df.columns and 'total_items subtotal' not in df.columns:
    df.rename(columns={'subtotal': 'subtotal'}, inplace=True)

df['created_at'] = pd.to_datetime(df['created_at'])
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'])

df['delivery_time_minutes'] = (df['actual_delivery_time'] - df['created_at']).dt.total_seconds() / 60.0
df = df.drop(columns=['actual_delivery_time'])

df = df[(df['delivery_time_minutes'] >= 1) & (df['delivery_time_minutes'] <= 180)]

df['created_hour'] = df['created_at'].dt.hour
df['created_day_of_week'] = df['created_at'].dt.dayofweek
df['created_month'] = df['created_at'].dt.month
df = df.drop(columns=['created_at'])

df['available_partners'] = df['total_onshift_partners'] - df['total_busy_partners']

df['market_id'] = df['market_id'].fillna(-1).astype(str)
df['store_primary_category'] = df['store_primary_category'].fillna('Unknown').astype(str)
df['order_protocol'] = df['order_protocol'].fillna(-1).astype(str)

num_cols = [
    'subtotal', 'num_distinct_items', 'min_item_price', 'max_item_price',
    'total_onshift_partners', 'total_busy_partners', 'total_outstanding_orders',
    'available_partners'
]

for col in num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

df.to_csv("../data/porter_eda_cleaned.csv", index=False)
print("Saved cleaned data to ../data/porter_eda_cleaned.csv")

Saved cleaned data to ../data/porter_eda_cleaned.csv


In [4]:
import pandas as pd

df_clean = pd.read_csv("../data/porter_eda_cleaned.csv")

print("Shape:", df_clean.shape)
print("Missing Values:\n", df_clean.isnull().sum())
print("\nTarget Summary:\n", df_clean['delivery_time_minutes'].describe())

Shape: (197283, 16)
Missing Values:
 market_id                   0
store_primary_category      0
order_protocol              0
total_items                 0
subtotal                    0
num_distinct_items          0
min_item_price              0
max_item_price              0
total_onshift_partners      0
total_busy_partners         0
total_outstanding_orders    0
delivery_time_minutes       0
created_hour                0
created_day_of_week         0
created_month               0
available_partners          0
dtype: int64

Target Summary:
 count    197283.000000
mean         47.533562
std          18.041246
min           1.683333
25%          35.066667
50%          44.316667
75%          56.316667
max         179.850000
Name: delivery_time_minutes, dtype: float64
